# Processamento de Língua Natural - EP (Testes)
Baseline: Regressão Logística com contagem TF-IDF

## Bibliotecas
Sklearn, Pandas

In [37]:
import numpy as np
import pandas as pd
from sklearn.base import clone
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import ParameterGrid, StratifiedKFold, cross_validate
from sklearn.pipeline import Pipeline

## Importação do Conjunto de Treino
O Pandas importa ambos os conjuntos de dados (treino e teste) como strings, para evitar erros de tipagem no momento da tokenização.

In [38]:
dados_treino = pd.read_excel("dados/train.xlsx", converters={'resp_text':str,'clarity':str})

X_treino, Y_treino = dados_treino["resp_text"].tolist(), dados_treino["clarity"].tolist()

In [39]:
dados_treino

,resp_text,clarity
0,"Prezado(a) Senhor(a), Esclarecemos que o Se...",c5
1,"Prezada cidadã, As informações sobre óbitos ...",c1
2,"Prezado Senhor Julio, A Ouvidoria-Geral da P...",c1
3,"Prezado(a) Senhor(a), Esclarecemos que o Se...",c234
4,"Senhor, O Serviço de Informações ao Cidadão d...",c234
...,...,...
20087,Prezado Sr. Luciano Wolff A sua solicitação ...,c1
20088,"PREZADO CIDADãO, A COORDENAçãO DO PROGRAMA FA...",c1
20089,"Prezado, Informamos que o Ministério da Agr...",c1
20090,"Prezado Senhor, Encaminhamos em anexo respos...",c1


## Tokenização (TF-IDF)

In [40]:
# Configuração base do TF-IDF
vect = TfidfVectorizer(
    max_df=0.95,
    sublinear_tf=True,
    max_features=20000
)

## Classificador (Regressão Logística)

In [41]:
# Classificador base
clf = LogisticRegression(
    max_iter=150,
    solver='saga',
    tol=1e-3,
    random_state=42
)

## Grid Search de TF-IDF + Regressão Logística

In [42]:
# Grid completo das combinações selecionadas
param_grid = {
    'ngram_range': [(1, 1), (1, 2)],
    'min_df': [2, 10],
    'C': [0.1, 0.5],
    'class_weight': [None, 'balanced']
}

# 2 folds para a busca, reduzindo o custo do grid.
# A melhor configuração será reavaliada com 10 folds abaixo.
cv_grid = StratifiedKFold(
    n_splits=2,
    shuffle=True,
    random_state=42
)

X_array = np.array(X_treino, dtype=object)
Y_array = np.array(Y_treino, dtype=object)

configuracoes = list(ParameterGrid(param_grid))
resultados_grid = []

for params in configuracoes:
    pipeline_grid = Pipeline([
        ('tfidf', TfidfVectorizer(
            ngram_range=params['ngram_range'],
            min_df=params['min_df'],
            max_df=0.95,
            sublinear_tf=True,
            max_features=20000
        )),
        ('clf', LogisticRegression(
            C=params['C'],
            class_weight=params['class_weight'],
            max_iter=150,
            solver='saga',
            tol=1e-3,
            random_state=42
        ))
    ])

    resultados = cross_validate(
        pipeline_grid,
        X_array,
        Y_array,
        cv=cv_grid,
        scoring='accuracy',
        return_train_score=True
    )

    treino = resultados['train_score'].mean()
    validacao = resultados['test_score'].mean()

    resultados_grid.append({
        **params,
        'acuracia_treino_media': treino,
        'acuracia_validacao_media': validacao,
        'desvio_padrao_validacao': resultados['test_score'].std(),
        'gap_treino_validacao': treino - validacao
    })

# Top 10 configurações
relatorio_grid = pd.DataFrame(resultados_grid).sort_values(
    by='acuracia_validacao_media',
    ascending=False
).reset_index(drop=True)

relatorio_grid.index = relatorio_grid.index + 1
relatorio_grid.index.name = 'rank'

top_10 = relatorio_grid.head(10)
display(top_10)

top_10.to_csv("grid_top10.csv")

# Melhor configuração encontrada
melhor = relatorio_grid.iloc[0]

print("Melhores hiperparâmetros:")
print({
    'ngram_range': melhor['ngram_range'],
    'min_df': melhor['min_df'],
    'C': melhor['C'],
    'class_weight': melhor['class_weight']
})

# Reavaliação do vencedor com 10 folds
pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(
        ngram_range=melhor['ngram_range'],
        min_df=int(melhor['min_df']),
        max_df=0.95,
        sublinear_tf=True,
        max_features=20000
    )),
    ('clf', LogisticRegression(
        C=float(melhor['C']),
        class_weight=melhor['class_weight'],
        max_iter=200,
        solver='saga',
        tol=1e-3,
        random_state=42
    ))
])

cv_final = StratifiedKFold(
    n_splits=10,
    shuffle=True,
    random_state=42
)

resultados_finais = cross_validate(
    pipeline,
    X_treino,
    Y_treino,
    scoring='accuracy',
    cv=cv_final,
    return_train_score=True
)

acuracia_validacao = resultados_finais['test_score'].mean()
acuracia_treino_cv = resultados_finais['train_score'].mean()

print(f"Acurácia Média da Validação Cruzada (10 folds): {acuracia_validacao:.2%}")
print(f"Acurácia Média de Treino nos Folds: {acuracia_treino_cv:.2%}")
print(f"Gap Treino-Validação: {(acuracia_treino_cv - acuracia_validacao):.2%}")

,C,class_weight,min_df,ngram_range,acuracia_treino_media,acuracia_validacao_media,desvio_padrao_validacao,gap_treino_validacao
rank,,,,,,,,
1,0.5,balanced,10,"(1, 2)",0.671412,0.453962,0.003235,0.217450
2,0.5,balanced,2,"(1, 2)",0.672158,0.453763,0.002041,0.218395
3,0.5,None,10,"(1, 2)",0.671262,0.453514,0.001792,0.217748
4,0.5,balanced,2,"(1, 1)",0.673203,0.452817,0.000796,0.220386
5,0.5,None,2,"(1, 2)",0.671660,0.452767,0.001642,0.218893
6,0.5,None,2,"(1, 1)",0.671163,0.452319,0.000398,0.218843
7,0.5,balanced,10,"(1, 1)",0.634581,0.449632,0.002289,0.184949
8,0.1,balanced,10,"(1, 2)",0.559277,0.448387,0.000050,0.110890
9,0.1,balanced,2,"(1, 2)",0.558979,0.448089,0.000348,0.110890


Melhores hiperparâmetros:
{'ngram_range': (1, 2), 'min_df': np.int64(10), 'C': np.float64(0.5), 'class_weight': 'balanced'}
Acurácia Média da Validação Cruzada (10 folds): 46.30%
Acurácia Média de Treino nos Folds: 64.06%
Gap Treino-Validação: 17.76%


## Treino do Modelo

In [43]:
pipeline.fit(X_treino, Y_treino)
print(f"Acurácia do Treino (modelo final): {pipeline.score(X_treino, Y_treino):.2%}")

Acurácia do Treino (modelo final): 63.57%


## Execução do Teste e Salvamento do Resultado

In [44]:
# Importação dos dados de teste
dados_teste = pd.read_excel("dados/test1.xlsx", converters={'resp_text':str,'clarity':str})
X_teste = dados_teste["resp_text"].tolist()

# Aplicação do modelo no teste
Y_previsto = pipeline.predict(X_teste)

In [45]:
teste_rotulado = pd.DataFrame({
    'resp_text': X_teste,
    'clarity': Y_previsto
})

teste_rotulado.to_excel("teste_rotulado.xlsx", index=False)